# 01 · Define & Explore — biosensor architectures, analyte + readout, sensor metrics

**Standard slot:** *define & explore.* **For Project 12 this means:** survey the two switch
architectures (allosteric / LOCKR-style conformational switches **and** split-reporter systems),
**choose your analyte and your readout**, write down the binder **and** sensor metrics + cutoffs, and
run a deterministic **mock** mini-run (binder → switch → ON/OFF) as your "hello-world" (D0).

Run `00_setup.ipynb` first in this session. A real binder campaign + switch modeling wants an
**A100** (see `MANUAL.md §2`); everything here runs on a no-GPU **mock** backend so you can build the
plumbing anywhere, then switch to the real backends on Colab Pro / A100.

> This project builds on the **binder-family workflow** (Project 06): the binder module is the same
> two-paradigm campaign. The new part is the **switch / split-reporter** that turns *binding* into
> *signal* — and reasoning about whether that signal has a usable **dynamic range**.

## Biosensor architectures (pick one to build on)

| Family | How binding → signal | Reporter | Key reference |
|--------|----------------------|----------|---------------|
| **Split-reporter** | binding reconstitutes a split enzyme/fluorophore | split-luciferase / **NanoBiT** (luminescence), split-FP (**FRET**) | Dixon 2016; Quijano-Rubio 2021 |
| **Conformational switch (LOCKR)** | analyte (or an exposed "key") displaces a **latch**, releasing a functional/reporter element | de novo cage + latch + key | Langan 2019 |

Both turn a **binding event** into a **measurable change**. The design decision that matters is
**mechanical coupling**: the binder must engage the analyte in a way that *toggles* the switch. A
great binder bolted to a switch that never moves is not a sensor.

## The metrics, precisely — binder metrics **and** sensor metrics

**Binder metrics** (same as the binder family; drive `design_type="binder"` filtering):

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the binder | thermostability / ΔG |
| **pae_interaction** | Å | AF2-Multimer error across the **binder–analyte interface** (key binder metric) | measured affinity |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency) | binding/function |
| rosetta_dG | REU | interface energy (more negative = stronger) | a guarantee it binds |
| shape complementarity | 0–1 | interface packing quality | epitope correctness |

**Sensor metrics** (new — describe the *transduction*, not just binding):

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| on_signal / off_signal | a.u. | modeled signal **with / without** analyte | a measured luminescence/FRET value |
| **dynamic_range** | fold | `on_signal / off_signal` — the headline sensor metric | a measured limit of detection |
| toggle_score | 0–1 | modeled separation between OFF and ON state conformations | that the real switch flips |
| background_leak | 0–1 | OFF-state signal leak (lower better) | assay background in the lab |

> The binder cutoffs are the shared `"binder"` cutoffs: **scRMSD ≤ 2.5, pLDDT ≥ 80,
> pae_interaction ≤ 10, rosetta_dG ≤ −30, sc ≥ 0.6.** The **sensor** is judged separately on
> **dynamic range** — and there is a real **affinity-vs-dynamic-range trade-off** (a too-tight binder
> can lock the switch ON regardless of analyte). A passing design is a **hypothesis** until a
> functional luminescence/FRET dose-response is run.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Choose the analyte + readout (STUDENT CHOICE)

**You pick the analyte.** Good capstone choices are small, structurally-characterized protein
biomarkers — e.g. a **cytokine** (inflammation) or a **cardiac marker** (e.g. a troponin subunit).
The catalog deliberately leaves this open: choose something with a verified RCSB structure and a clear
point-of-care motivation, and **mark the accession "candidate — verify on RCSB"** in Week 1
(`data/README.md`). Then pick a **readout**: luminescence (split-luciferase/NanoBiT) or FRET
(split-fluorophore).

Below we just *declare* an EXAMPLE analyte + epitope so the notebook runs end-to-end; **replace them
with your verified choice** (numbering depends on the PDB you clean).

In [ ]:
import biosensor_tools as bt

# --- STUDENT CHOICE: set these in Week 1 from your verified analyte structure ---
ANALYTE  = "ANALYTE"                 # e.g. your chosen cytokine / cardiac marker (cleaned target PDB)
# EXAMPLE epitope residues on the analyte — VERIFY/REPLACE from the analyte structure (data/README.md).
HOTSPOTS = bt.parse_hotspots("A12,A45,A60")   # EXAMPLE_DATA placeholder residues
READOUT  = "split_luciferase"        # one of: split_luciferase | nanobit | split_fluorophore_fret
SWITCH_FAMILY = "split_reporter"     # "split_reporter" (luminescence/FRET) or "lockr" (cage+latch)

print("analyte      :", ANALYTE, " (STUDENT CHOICE — verify the accession on RCSB)")
print("epitope/hotspots:", HOTSPOTS, " (EXAMPLE — replace with your verified residues)")
print("readout      :", READOUT)
print("switch family:", SWITCH_FAMILY)

## 2 · Mock hello-world: binder → switch → ON/OFF

`scripts/biosensor_tools.py` exposes the binder paradigms (`generate_binders_bindcraft`,
`generate_binders_rfdiffusion`, `af2_multimer`) **plus** the switch module (`design_switch`,
`integrate_binder_switch`, `model_two_state` / `two_state_readout`). The **mock** backend is
deterministic and GPU-free so you can develop the whole binding→signal plumbing. **Never report mock
numbers as real** — they are `SYNTHETIC` by construction (no real luminescence, no real LOD).

In [ ]:
# A few binders from each paradigm, scored by mock AF2-Multimer. All numbers are SYNTHETIC.
bc = bt.generate_binders_bindcraft(ANALYTE, HOTSPOTS, n=3, tool="mock")
rf = bt.generate_binders_rfdiffusion(ANALYTE, HOTSPOTS, n=3, tool="mock")
bt.score_designs(bc, tool="mock")
bt.score_designs(rf, tool="mock")

b = bc[0]
print("example binder design:")
print("  id   :", b.design_id, " len:", b.length, "aa")
print("  pae_interaction =", b.pae_interaction, " scrmsd =", b.scrmsd,
      " sc =", b.shape_complementarity, " (SYNTHETIC)")

In [ ]:
# Design a switch, integrate the binder, and reason about ON/OFF (all SYNTHETIC).
switch = bt.design_switch(scaffold="rfdiff_scaffold_01", reporter=READOUT,
                          family=SWITCH_FAMILY, n=1, tool="mock")[0]
construct = bt.integrate_binder_switch(b, switch, tool="mock")
bt.two_state_readout(construct, tool="mock")   # fills on/off/dynamic_range

print("switch   :", switch.switch_id, " toggle_score=", switch.toggle_score,
      " background_leak=", switch.background_leak, " (SYNTHETIC)")
print("construct:", construct.construct_id)
print("  on_signal =", construct.on_signal, " off_signal =", construct.off_signal,
      " dynamic_range =", construct.dynamic_range, " (SYNTHETIC)")
print("  LOD planning flag:", bt.estimate_lod(construct.dynamic_range, assay_cv=0.10))
print("\nReminder: switch tool='mock' -> 'bindcraft'/'rfdiffusion'/'af2' on Colab (A100). See MANUAL.md §2.")
print("dynamic_range here is SYNTHETIC — a real value needs a luminescence/FRET dose-response (nb 05).")

## 3 · Epitope-coverage proxy (does the binder engage the intended epitope?)

For a biosensor the binder must engage the analyte at the epitope you chose, so that binding couples
to the switch. `hotspot_overlap()` is a geometry proxy (fraction of the chosen epitope contacted) — a
teaching stand-in. Higher ⇒ the binder is engaging where you intended (not a guarantee of switching).

In [ ]:
for d in bc[:3]:
    ov = bt.hotspot_overlap(d.contact_residues, HOTSPOTS)
    print(f"{d.design_id}: contacts {d.contact_residues} -> epitope coverage = {ov} (SYNTHETIC)")

## Visualize a binder–analyte complex (py3Dmol)

Use this to eyeball a predicted binder–analyte complex (or, later, the integrated construct) once you
have a real PDB from AF2-Multimer / two-state modeling.

In [ ]:
import py3Dmol

def show_complex(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after a real AF2-Multimer / two-state prediction writes a complex PDB):
# show_complex("results/af2/top_complex.pdb")
print("show_complex(pdb_path) ready.")

## D0 checklist
- [ ] Analyte chosen (small protein biomarker) + accession marked **candidate — verify on RCSB**.
- [ ] Readout chosen (split-luciferase / NanoBiT luminescence, or split-FP FRET) + switch family.
- [ ] Epitope/hotspots derived from the analyte structure (not invented).
- [ ] One-paragraph definition of each **binder** and **sensor** metric **with** its "does not mean" note.
- [ ] Reproduced mock hello-world (binder → switch → ON/OFF) with metrics printed and flagged SYNTHETIC.
- [ ] Problem statement with measurable success criteria (e.g. target dynamic range) + controls
      (no-analyte, off-target); `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — the binder campaign + the switch module.